# Task 3 -- Ablation matrix

Each row adds or removes one kind of information relative to its neighbour,
so a change in score can be attributed. The rows are fitted by
`src/ablation.py`; run it first to produce the `results/ablation_*.csv` files.

    python src/ablation.py --build-raw   # once; needs data/embeddings/ (C-raw only)
    python src/ablation.py

| Row | Features | Question |
|---|---|---|
| A | rates only | time-series baseline |
| B | rates + Task 2 lexicons | does basic NLP help? |
| C1 / C2 | rates + GloVe / FastText axes | static embeddings |
| C3 / C4 | rates + FinBERT / SBERT axes | contextual; domain vs general |
| C-fbs | rates + FinBERT sentiment | why not just FinBERT sentiment? |
| C-raw | rates + PCA of raw embeddings (best encoder) | do axes beat dumping vectors? |
| C5 | best C + novelty and concentration | does news surprise add signal? |
| C6 / C6-stack | late fusion with / without the novelty gate | fusion strategy |
| drop-* | C5 minus one channel | which channel matters |

**Selection.** Every row picks its model family (logistic regression or
XGBoost), hyperparameters and decay half-life on validation macro-F1. "Best
encoder" is the C1-C4 row with the best validation macro-F1, so it can differ
by target. The test block is scored once.

**Uncertainty.** With 176 test days a 0.02 MCC change is noise. Every row
reports MCC(row) - MCC(A) with a 95% circular block-bootstrap CI (10-session
blocks, 2,000 resamples, both models scored on the same resampled days).

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
import viz

viz.apply_style()
pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 60)
print("project root:", ROOT)

In [ ]:
from anchors import CHANNELS

metrics = pd.read_csv(config.ABLATION_METRICS_CSV)
boot = pd.read_csv(config.ABLATION_BOOTSTRAP_CSV)
TARGETS = ["direction", "large_move"]
ORDER = (["A", "B", "C1", "C2", "C3", "C4", "C-fbs", "C-raw", "C5", "C6", "C6-stack"]
         + [f"drop-{c}" for c in CHANNELS])
print(f"{metrics.row.nunique()} rows x {metrics.target.nunique()} targets")

## 1. The matrix (test block)

In [ ]:
cols = ["description", "encoder", "family", "decay_halflife", "accuracy", "f1_macro",
        "mcc", "roc_auc", "d_mcc", "ci_low", "ci_high", "p_gt0"]

def matrix(target):
    t = metrics[(metrics.target == target) & (metrics.split == "test")]
    t = t.merge(boot[boot.target == target], on=["target", "row"]).set_index("row")
    return t.reindex([r for r in ORDER if r in t.index])[cols].round(3)

matrix("direction")

In [ ]:
matrix("large_move")

## 2. MCC difference against Model A

A dot right of zero beats the rates-only baseline on the test block. The bar is
the 95% block-bootstrap CI of that difference; a bar that crosses zero is not
distinguishable from no effect.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 7), sharey=True)
rows = [r for r in ORDER if r != "A"]
ypos = {r: len(rows) - 1 - i for i, r in enumerate(rows)}
for ax, target in zip(axes, TARGETS):
    b = boot[(boot.target == target) & boot.row.isin(rows)]
    y = b.row.map(ypos)
    ax.errorbar(b.d_mcc, y, xerr=[b.d_mcc - b.ci_low, b.ci_high - b.d_mcc],
                fmt="o", color=viz.BLUE, ecolor=viz.BLUE, elinewidth=1.5,
                markersize=6, capsize=0)
    ax.axvline(0, color=viz.TEXT_SECONDARY, lw=1)
    for edge in ("C1", "C5", f"drop-{CHANNELS[0]}"):   # group separators
        ax.axhline(ypos[edge] + 0.5, color=viz.GRID, lw=1)
    ax.set_title(target)
    ax.set_xlabel("test MCC minus Model A's")
    ax.grid(axis="y", visible=False)
axes[0].set_yticks(list(ypos.values()), list(ypos.keys()))
fig.suptitle("Difference in test MCC vs Model A, with 95% block-bootstrap CI", x=0.01, ha="left")
plt.tight_layout()
plt.show()

## 3. Does validation predict test?

Every choice in the matrix (encoder, family, half-life) is made on validation.
If validation and test scores are unrelated, those choices are close to random.
Late-fusion rows are left out because their meta-model is fitted on
validation, so their validation scores are in-sample.

In [ ]:
v = metrics[metrics.split == "validation"]
t = metrics[metrics.split == "test"]
m = v.merge(t, on=["target", "row"], suffixes=("_val", "_test"))
m = m[~m.val_in_sample_val]

fig, ax = plt.subplots(figsize=(6.6, 5.2))
for target, color in zip(TARGETS, [viz.BLUE, viz.ORANGE]):
    s = m[m.target == target]
    r = s[["mcc_val", "mcc_test"]].corr().iloc[0, 1]
    ax.scatter(s.mcc_val, s.mcc_test, s=40, color=color,
               edgecolor=viz.SURFACE, linewidth=1.5, label=f"{target} (r = {r:+.2f})")
ax.axhline(0, color=viz.TEXT_SECONDARY, lw=1)
ax.axvline(0, color=viz.TEXT_SECONDARY, lw=1)
ax.set_xlabel("validation MCC")
ax.set_ylabel("test MCC")
ax.set_title("Validation MCC vs test MCC, one dot per ablation row")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
# What validation chose, per target.
sel = (metrics[(metrics.split == "test") & metrics.row.isin(["A", "C1", "C2", "C3", "C4", "C5"])]
       [["target", "row", "encoder", "family", "decay_halflife", "params", "n_features"]])
sel.sort_values(["target", "row"]).reset_index(drop=True)

## 4. Which channel matters? (drop-one)

Change in test MCC when one channel's direction and attention scores are
removed from C5. A negative bar means the model got worse without the channel,
i.e. the channel was helping.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, target in zip(axes, TARGETS):
    t = metrics[(metrics.target == target) & (metrics.split == "test")].set_index("row")
    d = pd.Series({c: t.loc[f"drop-{c}", "mcc"] - t.loc["C5", "mcc"] for c in CHANNELS})
    ax.barh(list(d.index)[::-1], d.values[::-1], height=0.62,
            color=[viz.COLOR_DOWN if x < 0 else viz.COLOR_UP for x in d.values[::-1]])
    ax.axvline(0, color=viz.TEXT_SECONDARY, lw=1)
    ax.set_title(f"{target} (C5 on {t.loc['C5', 'encoder']})")
    ax.set_xlabel("test MCC without the channel minus C5's")
    ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

## 5. Findings

1. **Several text-based models improve direction prediction on the test block.**
   Relative to Model A (rates-only, MCC = -0.077), C1 (GloVe axes, +0.184,
   95% CI [+0.076, +0.303]), C2 (FastText axes, +0.220, CI [+0.008, +0.422]),
   C5 (SBERT axes + novelty, +0.208, CI [+0.050, +0.384]), and C6-stack
   (ungated late fusion, +0.245, CI [+0.118, +0.382]) have confidence
   intervals entirely above zero. The drop-geo and drop-trade rows also
   have intervals entirely above zero. These are comparisons against Model A,
   not pairwise comparisons between text models.

2. **Validation performance does not predict test performance.**
   Across the 18 ablation rows, validation and test MCC have correlations
   of -0.053 for direction and -0.032 for large moves. Both are effectively
   zero, suggesting that validation-based model selection did not reliably
   identify models that performed well on the held-out test period.

3. **Text features do not reliably improve large-move prediction.**
   Model A has an MCC of 0.115. No text-based row has a 95% bootstrap
   confidence interval entirely above zero relative to Model A. C5 has a
   slightly higher test MCC (0.127), but its difference is small and its
   confidence interval crosses zero.

4. **Semantic axes outperform raw embeddings descriptively.**
   For direction, the selected SBERT axes row (C4) has MCC 0.027, compared
   with -0.007 for raw SBERT PCA features. For large moves, the selected
   GloVe axes row (C1) has MCC 0.112, compared with -0.018 for raw GloVe
   PCA features. These are descriptive comparisons; no pairwise bootstrap
   test was conducted.

5. **The additional fusion and novelty features have mixed results.**
   C5 improves direction MCC relative to its selected encoder-only row
   (C4), but does not reliably improve large-move prediction. Ungated
   late fusion (C6-stack) has the highest direction MCC (0.168), while
   gated late fusion (C6) scores 0.040. Neither fusion approach improves
   large-move MCC over Model A.

6. **FinBERT sentiment does not show a reliable improvement.**
   The C-fbs row has a direction MCC of -0.024 and a large-move MCC of
   0.004. Neither result provides evidence of improvement over Model A.

7. **Drop-one results suggest that channel contributions vary by target.**
   For direction, all seven drop-one rows have positive MCC relative to
   Model A, and drop-geo and drop-trade have confidence intervals entirely
   above zero. For large moves, drop-geo has a confidence interval entirely
   below zero relative to Model A. However, these intervals do not directly
   test the effect of removing a channel from C5; that would require
   pairwise bootstrap comparisons against C5.

**Bottom line for the report.** Text-based features show promising
improvements for direction prediction on this test block, particularly
semantic axes and ungated late fusion. However, validation-test correlations
are close to zero, and the large-move experiments show no reliable
improvement over the rates-only baseline. Results should be interpreted
cautiously because they come from a single held-out test period and multiple
model comparisons.
